# ACE reproduction on Formula (FTEC5660)
Reproduces Table 2 of *Agentic Context Engineering* (ICLR 2026) with the DeepSeek API instead of DeepSeek-V3.1 on SambaNova,
then runs one modification (A, de-duplication on) and one ablation (C, no ground truth).
Put the key in Colab Secrets as `DEEPSEEK_API_KEY`, or type it when asked. A CPU runtime is enough.
The setup cell clones the fork, or resets an existing checkout to it, so it runs exactly what is on GitHub.


In [ ]:
import getpass, pathlib, subprocess
%cd /content
# update in place: results/ and .env are git-ignored, so earlier runs survive a re-run of this cell
!if [ -d ace/.git ]; then git -C ace fetch -q && git -C ace reset -q --hard origin/main; else git clone -q https://github.com/baimingyang98/ace.git; fi
%cd /content/ace
!git log --oneline -1
!pip -q install "openai>=1.0.0" python-dotenv tiktoken sentence-transformers faiss-cpu

key = None
try:
    from google.colab import userdata
    key = userdata.get("DEEPSEEK_API_KEY")
except Exception:
    pass
if not key:
    key = getpass.getpass("DEEPSEEK_API_KEY: ")
pathlib.Path(".env").write_text(f"DEEPSEEK_API_KEY={key}\n")
print("key written to .env (not printed)")

MODEL = "deepseek-flash"    # or "deepseek-v4-pro", about 4x the price (a full pro run cost more than 20 yuan)
MAX_ROUNDS = 5              # paper section 4.2 uses 5 Reflector rounds; the repo default is 3
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()


In [ ]:
import csv, glob, json, os, re, shutil, time
from importlib.metadata import version
from utils import count_tokens

# Yuan per 1M tokens (input on a cache miss, output), off-peak, from api-docs.deepseek.com/zh-cn/quick_start/pricing
# (read 2026-10-24). Peak hours, Beijing time 9-12 and 14-18 on weekdays, cost 2x; cache hits cost far less.
PRICES = {"deepseek-flash": (1.0, 4.0), "deepseek-v4-pro": (4.5, 13.5)}
ROOT = "/content/ace"
EXPORT = "/content/ace_export"   # small files of every finished run; zipped to /content/ace_results.zip


def run_group(paths, extra=(), env=None, model=MODEL, rounds=MAX_ROUNDS):
    """Start one online Formula run per save path at the same time, wait for all, then export and summarize
    each. The API calls dominate, so parallel runs barely slow each other. Finished save paths are skipped."""
    procs = {}
    for path in paths:
        if glob.glob(f"{ROOT}/{path}/ace_run_*/final_results.json"):
            print("already done:", path)
            continue
        os.makedirs(f"{ROOT}/{path}", exist_ok=True)
        log_path = f"{ROOT}/{path}/stdout.log"
        if os.path.exists(log_path):  # keep the log of an earlier, unfinished attempt
            os.rename(log_path, log_path.replace(".log", f"_{int(os.path.getmtime(log_path))}.log"))
        cmd = ["python", "-m", "eval.finance.run", "--task_name", "formula", "--mode", "online",
               "--api_provider", "deepseek", "--generator_model", model, "--reflector_model", model,
               "--curator_model", model, "--max_num_rounds", str(rounds), "--save_path", path, *extra]
        procs[path] = subprocess.Popen(cmd, cwd=ROOT, stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
                                       env={**os.environ, **(env or {})})
    t0 = time.time()
    try:
        for path, proc in procs.items():
            proc.wait()
            print(f"{path}: exit {proc.returncode} after {(time.time() - t0) / 60:.1f} min")
    except KeyboardInterrupt:
        # the runs are separate processes: without this they keep running, and paying, after an interrupt
        for proc in procs.values():
            proc.terminate()
        print("interrupted: all runs of this group were stopped")
        raise
    rows = []
    for path in paths:
        export(path)
        rows.append(summarize(path, model))
    failed = [path for path, proc in procs.items() if proc.returncode != 0]
    if failed:
        errors = {p: (re.findall(r"Error code: \d+.*", open(f"{ROOT}/{p}/stdout.log").read()) or ["see stdout.log"])[-1][:300]
                  for p in failed}
        raise RuntimeError(f"failed runs: {errors}")
    return rows


def export(save_path):
    """Turn the per-call logs of the latest run into calls.csv, copy everything else under save_path to EXPORT
    and rebuild the zip. A runtime that is later recycled then loses nothing that has finished."""
    runs = sorted(glob.glob(f"{ROOT}/{save_path}/ace_run_*"))
    if not runs:
        return
    rows = []
    for p in glob.glob(f"{runs[-1]}/detailed_llm_logs/*.json"):
        c = json.load(open(p))
        rows.append({k: c.get(k) for k in ("role", "call_id", "timestamp", "prompt_num_tokens",
                                           "response_num_tokens", "call_time", "total_time")}
                    | {"error": (c.get("error") or "")[:200]})
    if rows:
        rows.sort(key=lambda r: r["timestamp"] or "")
        with open(f"{runs[-1]}/calls.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0]))
            w.writeheader()
            w.writerows(rows)
    shutil.rmtree(f"{EXPORT}/{save_path}", ignore_errors=True)
    shutil.copytree(f"{ROOT}/{save_path}", f"{EXPORT}/{save_path}", ignore=shutil.ignore_patterns("detailed_llm_logs"))
    zipped = subprocess.run("rm -f /content/ace_results.zip && zip -qr /content/ace_results.zip .", shell=True,
                            cwd=EXPORT, capture_output=True, text=True)
    size = os.path.getsize("/content/ace_results.zip") / 1e6 if os.path.exists("/content/ace_results.zip") else 0
    print(f"exported {save_path} -> /content/ace_results.zip ({size:.1f} MB)", zipped.stderr.strip())


def summarize(save_path, model=MODEL):
    """Accuracy, calls, tokens, cost, silent failures and de-duplication of the latest run under save_path.
    Per-window accuracies and the wrong question indices are printed too, so the notebook output keeps them.
    A crashed run has no final_results.json; then the base accuracy comes from initial_test_results.json and
    the ACE accuracy is the cumulative online accuracy at the last finished window."""
    run_dir = sorted(glob.glob(f"{ROOT}/{save_path}/ace_run_*"))[-1]
    log = open(f"{ROOT}/{save_path}/stdout.log").read()
    final = f"{run_dir}/final_results.json"
    res = json.load(open(final)) if os.path.exists(final) else {}
    initial = json.load(open(f"{run_dir}/initial_test_results.json")) if os.path.exists(
        f"{run_dir}/initial_test_results.json") else {}
    online = json.load(open(f"{run_dir}/test_results.json")) if os.path.exists(f"{run_dir}/test_results.json") else {}
    if not res:
        res["initial_test_results"] = initial.get("test_results", {})
        windows = re.findall(r"Cumulative test accuracy so far: ([\d.]+) \((\d+) samples\)", log)
        if windows:
            res["online_test_results"] = {"accuracy": float(windows[-1][0]), "total": int(windows[-1][1])}
    calls = [json.load(open(p)) for p in glob.glob(f"{run_dir}/detailed_llm_logs/*.json")]
    ok = [c for c in calls if "prompt_num_tokens" in c]
    merges = [tuple(map(int, m)) for m in re.findall(r"\[ANALYZER\] merge call tokens: (\d+) (\d+)", log)]
    tin = sum(c["prompt_num_tokens"] for c in ok) + sum(m[0] for m in merges)
    tout = sum(c["response_num_tokens"] for c in ok) + sum(m[1] for m in merges)
    pin, pout = PRICES[model]
    playbook = f"{run_dir}/final_playbook.txt"
    row = {
        "run": save_path,
        "complete": os.path.exists(final),
        "online_samples_scored": res.get("online_test_results", {}).get("total"),
        "base_acc": res.get("initial_test_results", {}).get("accuracy"),
        "ace_acc": res.get("online_test_results", {}).get("accuracy"),
        "calls": len(calls) + len(merges),
        **{r: sum(c["role"] == r for c in calls) for r in ("generator", "reflector", "curator")},
        "merge_calls": len(merges),
        "merge_parse_failures": log.count("Failed to parse merged bullet"),
        "failed_or_empty": sum("error" in c for c in calls),
        "final_playbook_tokens": count_tokens(open(playbook).read()) if os.path.exists(playbook) else None,
        "input_Mtok": round(tin / 1e6, 2), "output_Mtok": round(tout / 1e6, 2),
        "yuan_offpeak_no_cache": round(tin / 1e6 * pin + tout / 1e6 * pout, 2),
    }
    print(json.dumps(row, indent=1))
    print("window accuracies:", [w["window_accuracy"] for w in online.get("test_results", {}).get("window_results", [])])
    print("base wrong:", sorted(e["index"] for e in initial.get("error_log", {}).get("errors", [])))
    print("ACE wrong:", sorted(e["global_index"] for e in online.get("test_error_log", {}).get("errors", [])))
    return row


print("model:", MODEL, "| provider: deepseek | base_url: https://api.deepseek.com | temperature: 0.0 (fixed in llm.py)",
      "| max_tokens: 4096 | thinking:", os.environ.get("DEEPSEEK_THINKING", "disabled"),
      "| commit:", COMMIT, "| openai", version("openai"))


## 1. Smoke test


In [ ]:
# One call through the repo's own wrapper (key, provider, thinking off), and the de-duplication dependencies.
# Loading the embedding model here also downloads it once, before three runs would fetch it at the same time.
from dotenv import load_dotenv
load_dotenv(".env", override=True)
from utils import initialize_clients
from llm import timed_llm_call
from ace.core import bulletpoint_analyzer

client = initialize_clients("deepseek")[0]
text, info = timed_llm_call(client, "deepseek", MODEL, "What is 15% of 300,000? Reply with the number only.",
                            role="generator", call_id="smoke", max_tokens=50)
raw = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Say OK."}], max_tokens=5,
                                     extra_body={"thinking": {"type": "disabled"}})
print(MODEL, "| answer:", text.strip(), "| served model:", raw.model,
      "| reasoning_content:", getattr(raw.choices[0].message, "reasoning_content", None))
print("de-duplication available:", bulletpoint_analyzer.DEDUP_AVAILABLE)
from sentence_transformers import SentenceTransformer
SentenceTransformer("all-mpnet-base-v2")
print("embedding model cached")


## 2. Earlier runs
Recover runs still on this runtime, list leftover processes, and load the numbers of the baseline (2026-10-24) and of ablation C (2026-10-25).


In [ ]:
# Only if this runtime still holds earlier runs: export them so they can be downloaded
for path in sorted(glob.glob(f"{ROOT}/results/*")):
    name = path[len(ROOT) + 1:]
    if glob.glob(f"{path}/ace_run_*"):
        export(name)
        print("exported", name)
!ls -lh /content/ace_results.zip 2>/dev/null || echo "no earlier runs on this runtime"


In [ ]:
# Leftover runs and progress. Runs started before an interrupt can keep going in the background.
# The gap from a Curator call to the next Generator call is the de-duplication time plus one call (about 2 s).
from datetime import datetime
print(subprocess.run("ps -eo pid,etime,pcpu,args | grep 'eval.finance.run' | grep -v grep", shell=True,
                     capture_output=True, text=True).stdout or "no ACE process running")


def stamp(name):
    return datetime.strptime(re.search(r"(\d{8}_\d{6}_\d{3})\.json$", name).group(1), "%Y%m%d_%H%M%S_%f")


for path in sorted(glob.glob(f"{ROOT}/results/*")):
    runs = sorted(glob.glob(f"{path}/ace_run_*"))
    if not runs or not os.path.exists(f"{path}/stdout.log"):
        continue
    log = open(f"{path}/stdout.log").read()
    windows = re.findall(r"Cumulative test accuracy so far: ([\d.]+) \((\d+) samples\)", log)
    sizes = re.findall(r"Analyzing (\d+) bulletpoints", log)
    files = os.listdir(f"{runs[-1]}/detailed_llm_logs")
    cur = {m.group(1): stamp(f) for f in files if (m := re.match(r"curator_online_train_s_(\d+)_\d{8}", f))}
    post = {m.group(1): stamp(f) for f in files if (m := re.match(r"generator_online_train_s_(\d+)_post_curate_", f))}
    gaps = sorted((int(k), round((post[k] - cur[k]).total_seconds())) for k in cur if k in post)
    print(path[len(ROOT) + 1:], "| finished:", bool(glob.glob(f"{runs[-1]}/final_results.json")),
          "| last window:", windows[-1] if windows else None, "| bullets at last de-dup:", sizes[-1] if sizes else None,
          "| (step, seconds):", [g for g in gaps if g[0] % 25 == 0] + gaps[-1:])
# stop leftover runs with:  !pkill -f eval.finance.run


In [ ]:
# Baseline, 2026-10-24, commit 528c9b9: online ACE on all 200 Formula questions, deepseek-flash, 5 rounds.
# Copied from the notebook output of that session; the run files were not on the runtime the next day.
baseline = [
    {"run": "baseline_run1", "base_acc": 0.735, "ace_acc": 0.785, "calls": 1253, "input_Mtok": 7.99, "output_Mtok": 0.42, "yuan_offpeak_no_cache": 9.68},
    {"run": "baseline_run2", "base_acc": 0.725, "ace_acc": 0.790, "calls": 1261, "input_Mtok": 8.76, "output_Mtok": 0.43, "yuan_offpeak_no_cache": 10.48},
    {"run": "baseline_run3", "base_acc": 0.715, "ace_acc": 0.810, "calls": 1245, "input_Mtok": 8.65, "output_Mtok": 0.39, "yuan_offpeak_no_cache": 10.21},
]
# Ablation C, 2026-10-25, commit 99a2e51, copied from that session's output (used if rows_c is not in memory)
nogt = [
    {"run": "nogt_run1", "base_acc": 0.710, "ace_acc": 0.825, "calls": 1371, "input_Mtok": 9.56, "output_Mtok": 0.55, "yuan_offpeak_no_cache": 11.78},
    {"run": "nogt_run2", "base_acc": 0.710, "ace_acc": 0.800, "calls": 1398, "input_Mtok": 11.24, "output_Mtok": 0.61, "yuan_offpeak_no_cache": 13.66},
    {"run": "nogt_run3", "base_acc": 0.720, "ace_acc": 0.795, "calls": 1378, "input_Mtok": 8.91, "output_Mtok": 0.54, "yuan_offpeak_no_cache": 11.08},
]


## 3. Modification A: grow-and-refine de-duplication, 3 runs in parallel (about 50 minutes)


In [ ]:
# Modification A: switch on the grow-and-refine de-duplication of section 3.2 (off in the repo's default run).
# After each Curator step, bullets with embedding similarity >= 0.90 (all-mpnet-base-v2) are merged by the LLM.
# One thread per run: the embedding model runs on the CPU, and three runs with all threads each fight for it.
rows_a = run_group([f"results/formula_online_{MODEL}_r{MAX_ROUNDS}_dedup_run{i}" for i in (1, 2, 3)],
                   extra=("--use_bulletpoint_analyzer",),
                   env={"OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "TOKENIZERS_PARALLELISM": "false"})


Download `/content/ace_results.zip` from the file browser on the left after each section. It holds every finished run (all files except the per-call logs, which are summarized in `calls.csv`).


## 4. Ablation C: no ground-truth labels, 3 runs in parallel (about 50 minutes; done on 2026-10-25)


In [ ]:
# Ablation C: the Reflector and Curator get no ground-truth answer (paper Table 2, Formula online: 78.5).
# Note: the loop still checks each answer against the label to decide whether to reflect again, and tells the
# Reflector whether the answer matched, so a correctness signal remains.
rows_c = run_group([f"results/formula_online_{MODEL}_r{MAX_ROUNDS}_nogt_run{i}" for i in (1, 2, 3)],
                   extra=("--no_ground_truth",))


Download `/content/ace_results.zip` from the file browser on the left after each section. It holds every finished run (all files except the per-call logs, which are summarized in `calls.csv`).


## 5. Summary


In [ ]:
import pandas as pd
groups = {"baseline": baseline, "A: de-duplication": globals().get("rows_a", []), "C: no GT": globals().get("rows_c", nogt)}
summary = []
for name, rows in groups.items():
    if not rows:
        continue
    df = pd.DataFrame(rows)
    gain = df.ace_acc - df.base_acc
    summary.append({"config": name, "runs": len(df),
                    "base": f"{df.base_acc.mean():.3f} ± {df.base_acc.std():.3f}",
                    "ACE": f"{df.ace_acc.mean():.3f} ± {df.ace_acc.std():.3f}",
                    "gain": f"{gain.mean():+.3f} ± {gain.std():.3f}",
                    "input_Mtok": round(df.input_Mtok.mean(), 2),
                    "yuan_per_run": round(df.yuan_offpeak_no_cache.mean(), 2)})
print(pd.DataFrame(summary).to_string(index=False))
print("\npaper (DeepSeek-V3.1, Table 2, Formula): base 0.675, ACE online 0.765 with GT, 0.785 without GT")
